# `MT.PHY3` Maschinendynamik: **Dynamik Pendelwagen**

## Einleitung

In den ersten Wochen haben Sie gelernt, die Dynamik (Kinematik und Kinetik) ebener Systeme zu beschreiben.
Die bisherigen Aufgaben im Unterricht und Praktikum waren dabei so gestellt, dass sie rein analytisch lösbar waren – d. h., Sie konnten mit Papier und Stift zur Lösung gelangen.
Probleme in der Praxis sind jedoch meist so komplex, dass analytische Ansätze entweder nicht anwendbar oder nicht zielführend sind.
Zur Lösung praktischer Problemstellungen werden daher sehr oft numerische Methoden eingesetzt.

In diesem Praktikum erlernen Sie mittels numerischer Methoden und Programmierung die _Bewegungsgleichungen von dynamischen Starrkörpersystemen zu lösen_.
Konkret berechnen Sie den Bewegungsablauf von einem Wagen mit Pendel, der auf einen elastischen Puffer auffährt.
In dieser Aufgabe sind als Ausgangspunkt die Bewegungsdifferentialgleichungen des mechanischen Systems bereits gegeben.
Um die Bewegungsgleichungen numerisch zu integrieren, müssen diese in ein System von Differentialgleichungen 1. Ordnung überführt werden.
In diesem Notebook durchlaufen Sie Schritt für Schritt das Vorgehen zur Lösung der Bewegungsgleichung, analyisieren die Resultate und stellen diese anschaulich dar.

<span style="color:Orange">Bitte bearbeiten Sie dieses Jupyter Notebook zu zweit oder zu dritt.
Arbeiten Sie sich von oben nach unten vor und ergänzen Sie die Stellen, die mit `<edit>` markiert sind (Anweisungen sind wie hier orange eingefärbt).
Gehen Sie insbesondere vertieft auf die **Verständnisfragen** ein.</span>

Das Ziel dieses Praktikums liegt in der praktischen Anwendung der numerischen Methoden, die im [Python](https://www.python.org/doc/)-Ökosystem mit verschiedenen Paketen (z.B., [NumPy](https://numpy.org/doc/stable/), [SciPy](https://docs.scipy.org/doc/scipy/), [Matplotlib](https://matplotlib.org/stable/index.html), ...) schlüsselfertig zur Verfügung stehen.
Die Funktionsweise und Grenzen dieser Algorithmen werden Sie im 4. Semester in der Vorlesung [Numerik (XXM6.NUM)](https://storagemkbdata.blob.core.windows.net/soe/moddescversions/Modulbeschreibung_t.BA.XXM6.NUM.22HS.pdf) erlernen.

## Lernziele

Sie können
 * die Bewegungsdifferentialgleichungen eines dynamischen Starrkörpersystems in die Zustandsform überführen.
 * das Anfangswertproblem durch numerische Integration der Zustandsform lösen.
 * die Resultate auswerten, interpretieren und anschaulich darstellen.

## Module

In [ ]:
import numpy as np                                          # numerical computing
from scipy.integrate import solve_ivp                       # solve an initial value problem
from scipy.interpolate import CubicSpline                   # spline interpolation
import matplotlib.pyplot as plt                             # ploting
import matplotlib.animation as animation                    # animation
from IPython.display import HTML                            # movies within the notebook

In [ ]:
%matplotlib inline

## Inhalt

 0. <a href="#Einmassenschwinger">Lineare Schwingung mit $f=1$ (Einführungsbeispiel)</a>
 1. [Kinetik Pendelwagen](#Pendelwagen)

<a id="Einmassenschwinger"></a>
## 0. Einmassenschwinger (Einführungsbeispiel)

Bevor Sie sich mit dem komplexeren Problem des Pendelwagens beschäftigen, analysieren Sie als Einstieg die Ihnen vertraute Kinetik eines Einmassenschwingers.

### 0.1 System

<img src="kinetik_einmassenschwinger.svg" width="600pt">

Der abgebildetet Einmassenschwinger mit Masse $m$, Federkonstante $c$, Dämpferkonstante $d$ und Kraftanregung $F(t)$ mit Freiheitsgrad $f=1$ gehorcht der Differentialgleichung
\begin{equation}\tag{$\ast$}
    m\,\ddot{x} + d\,\dot{x} + c\,x = F(t).
\end{equation}

### 0.2 Parameter

In unserem Beispiel rechnen wir mit folgenden Werten:
| Parameter               | Variable | Wert                   |
|:----------------------- |:--------:|:----------------------:|
| Masse                   |  $m$     | $1\,\mathrm{kg}$       |
| Dämpferkonstante        |  $d$     | $0.2\,\mathrm{Ns/m}$   |
| Federkonstante          |  $c$     | $100\,\mathrm{N/m}$    |

In [ ]:
m = 1                                                       # mass [kg]
d = 0.2                                                     # damping coefficient [Ns/m]
c = 100                                                     # spring coefficient [N/m]

Wir betrachten hier nur das _homogene System_ (also ohne äussere Kraftanregung) und setzen $F(t) = 0$

> _**Hinweis**_: In Python kann man mit [`lambda`](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions) kurz und bündig eine _einzeilige_ Funktion erzeugen. Alternativ könnte die Funktion auch mit [`def`](https://docs.python.org/3/tutorial/controlflow.html#defining-functions) definiert werden.

In [ ]:
F = lambda t: 0

Als Anfangsbedingung wählen wir die Anfangsauslenkung $x_0 = 5\,\mathrm{mm}$ und die Anfangsgeschwindigkeit $v_0 = 100\,\mathrm{mm/s}$.

In [ ]:
x0 = 5e-3                                                   # initial position [m]
v0 = 100e-3                                                 # initial velocity [m/s]

### 0.3 Homogene Lösung (händisch mit Fallunterscheidung für $D$)

Bevor wir die Lösung numerisch berechnen, bestimmen Sie die homogene Lösung analytisch.

<span style="color:Orange">Bestimmen Sie die dazu Eigenkreisfrequenz des ungedämpften Systems $\omega_0$, den Dämpfungswert $\delta$ sowie das Lehr'sche Dämfungsmass $D$.</span>

In [ ]:
ω0 = <edit>                                                 # angular eigenfrequency of the undamped system [rad/s]
δ = <edit>                                                  # damping factor [1/s]
D = <edit>                                                  # damping ratio [-] according to Lehr

In [ ]:
print(f"D = {D:.8g}           Lehr'sches Dämpfungsmass")

<span style="color:Orange">Ist das System</span>
 1. <span style="color:Orange">unterkritisch,</span>
 2. <span style="color:Orange">kritisch oder</span>
 3. <span style="color:Orange">überkritisch</span>
<span style="color:Orange">gedämpft?</span>

<span style="color:Orange">Bestimmen Sie die Pseudokreisfrequenz $\omega$, die Pseudofrequenz $f$ sowie die Pseudeperiode $T$.

In [ ]:
ω = <edit>                                                  # angular frequency of the damped system [rad/s]
f = <edit>                                                  # frequency of the damped system [Hz]
T = <edit>                                                  # pseudo period [s]

<span style="color:Orange">Geben Sie die analytische Lösung $x_h(t)$ der homogenen Differentialgleichung $(\ast)$ an, die die Anfangsbedingungen $x_h(0) = x_0$ und $\dot{x}_h(0) = v_0$ erfüllt.
Berechnen Sie $x_h(t)$ und $\dot{x}_h(t)$ diskret über die Zeitspanne von $t \in [0, 10\,T].$</span>

In [ ]:
t = np.linspace(0, 10*T, 1000)                              # time vector

In [ ]:
C1 = <edit>
C2 = <edit>
xh = <edit>
xh_t = <edit>

### 0.4 Numerische Lösung (explizite Integration in Zustandsform)

Die Bewegungsgleichung des Einmassenschwingers $(\ast)$ ist _eine lineare gewöhnliche Differentialgleichung 2. Ordnung_. 
Für die numerische Lösung mit expliziter Zeitintegration (z. B. mit dem [Runge–Kutta-Verfahren](https://de.wikipedia.org/wiki/Runge-Kutta-Verfahren)) muss die Differntialgleichung 2. Ordnung in ein System von Differntialgleichungen 1. Ordnung überführt werden.
Das geschieht mit folgendem _**Trick**_:
Es wird $z:= \dot{x}$ als unabhängige Variable für die Geschwindigkeit $\dot{x}$ gesetzt.
Zusammen mit $(\ast)$ lässt sich damit _ein System von zwei Differentialgleichungen 1. Ordnung_ angeben
\begin{align}
    \dot{x} &= z \\
    m\,\dot{z} + d\,z + c\,x &= F(t).
\end{align}
Büschelt man um, folgt für das lineare System die Zustandsform
\begin{equation}\tag{$\ast\ast$}
    \underbrace{\begin{pmatrix}\dot{x} \\ \dot{z} \end{pmatrix}}_\dot{\mathbfit{y}} 
        = \underbrace{\begin{pmatrix} 0 & 1 \\ -c/m & -d/m \end{pmatrix}}_\mathbf{A} 
        \underbrace{\begin{pmatrix}x \\ z \end{pmatrix}}_\mathbfit{y} + \underbrace{\begin{pmatrix}0 \\ F(t)/m \end{pmatrix}}_{\mathbfit{b}(t)},
\end{equation}
oder kurz
\begin{equation}\tag{$\ast\ast$'}
    \dot{\mathbfit{y}}= \underbrace{\mathbf{A}\,\mathbfit{y} + \mathbfit{b}(t)}_{\mathbfit{f}(t, \mathbfit{y})}.
\end{equation}
Darin definieren wir den Zustandsvektor $\mathbfit{y} = \begin{pmatrix} x \\ z \end{pmatrix}$, die Systemmatrix $\mathbf{A}$ und den Störterm $\mathbfit{b}(t)$.

Gleichung $(\ast\ast)$ ist _ein System von zwei gewöhnlichen Differentialgleichungen 1. Ordnung_, das durch explizite Zeitintegration numerisch gelöst werden kann.

<span style="color:Orange">Definieren Sie die Systemmatrix `A`.</span>

In [ ]:
A = np.array([
    [<edit>, <edit>],
    [<edit>, <edit>]
    ])

<span style="color:Orange">Definieren Sie die Funktion `b(t)` für den Störterm $\mathbfit{b}(t)$.</span>

In [ ]:
b = lambda t: np.array([<edit>, <edit>])

<span style="color:Orange">Definieren Sie die Funktion `f(t, y)`, die explizit die Zeitableitung $\dot{\mathbfit{y}} = \mathbfit{f}(t,\mathbfit{y})$ und damit die Differentialgleichung beschreibt.</span>

> _**Hinweis**_: Eine _Matrixmultiplikation_ wie $\mathbf{A}\,\mathbfit{y}$ wird in NumPy mit dem [`@`-Operator](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) ausgeführt.

In [ ]:
def f(t, y):
    """Differential equation in first-order form (state)
    
        y_t = f(t, y)       with y := [x, x_t]
    """
    return <edit>

Das Anfangswertproblem $(\ast)$ wird durch numerische Integration von $\dot{\mathbfit{y}} = \mathbfit{f}(t,\mathbfit{y})$ gelöst.
Die explizite Zeitintegration erfolgt mit der Funktion [`solve_ivp()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).
Als Parameter zu übergeben sind die zu integrierende explizite Funktion `f`, das Zeitinterval `t_span`, über as integriert wird, sowie der Anfangszustand `y0`.

> _**Hinweis**_: Standarmässig wird in [`solve_ivp()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) für die numerische Integration das [explizite Runge–Kutta-Verfahren der Ordnung 5(4)](https://de.wikipedia.org/wiki/Runge-Kutta-Verfahren) verwendet.

In [ ]:
y0 = np.array([x0, v0])
t_span = [min(t), max(t)]
result = solve_ivp(f, t_span, y0)

Das Resultat enthält die diskreten Zeitpunkte $t_i$ und zugehörigen Werte des Zustandsvektors $\mathbfit{y}_i = \begin{pmatrix} x_i \\ \dot{x}_i \end{pmatrix}$ .

In [ ]:
_t = result.t
_x, _x_t = result.y

In [ ]:
fig0, axs = plt.subplots(2, 1, num='Einmassenschwinger', clear=True)
axs[0].plot(t, xh, 'g-', alpha=.5, label='analytisch')
axs[0].plot(_t, _x, 'g.', label='numerisch')
axs[0].plot(0, x0, 'ko', label='Anfangsauslenkung')
axs[0].set_ylabel(r'Auslenkung  $x$  [m]')
axs[0].legend(loc='upper right')
axs[1].plot(t, xh_t, 'b-', alpha=.5, label='analytisch')
axs[1].plot(_t, _x_t, 'b.', label='numerisch')
axs[1].plot(0, v0, 'ko', label='Anfangsgeschwindigkeit')
axs[1].set_ylabel(r'Geschwindigkeit  $\dot{x}$  [m/s]')
axs[1].set_xlabel(r'Zeit  $t$  [s]');
axs[1].legend(loc='upper right');

 * <span style="color:Orange">Wie gut passt die numerisch berechnete Lösung zur analytischen Lösung?</span>
 * <span style="color:Orange">Was fällt bei der numerischen Lösung auf?</span>

### 0.5 Eigenwerte der Systemmatrix $\mathbf{A}$

<span style="color:Orange">Berechnen Sie die Eigenwerte $\lambda_i$ (und Eigenvektoren $\mathbfit{u}_i$) der Systemmatrix $\mathbf{A}$ mit der Funktion [`np.linalg.eig()`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html#numpy.linalg.eig).</span>

In [ ]:
λ, U = <edit>

<span style="color:Orange">Vergleichen Sie die Eigenwerte $\lambda$ der Systemmatrix mit der Pseudokreisfrequenz $\omega$ und dem Dämpfungsfaktor $\delta$.
Was stellen Sie fest?</span>

In [ ]:
print(f"λ = {λ}")

In [ ]:
print(f"δ = {δ}, ω = {ω}")

<a id="Pendelwagen"></a>
## 1. Kinetik Pendelwagen

### 1.1 System

In dieser Aufgabe berechnen Sie die Dynamik des abgebildeten Pendelwagens.
Der Pendelwagen besteht aus einem Quader (Masse $m_S$, Breite $W$, Höhe $H$) und einem Pendel (Masse $m_P$ in $P$, Länge $L$), das am Massenmittelpunkt $S$ des Quaders drehend gelagert ist.
Der Quader sei auf der Unterlage so geführt, dass sich dieser reibungsfrei in horziontaler Richtung bewegen kann.
An der linksseitigen Wand ist ein Puffer befestigt, der als Feder (Federkonstante $c$, entspannte Länge $l_0$) modelliert wird.
Die Feder kann nur eine Druckkraft aufnehmen, die wie eingezeichnet als $F_c(x)$ wirkt.
Die Koordinaten $x$ und $\varphi$ beschreiben die Auslenkung von Quader und Pendel.
Es wirkt Gravitation mit Erdbeschleunigung $g$.

<img src="kinetik_pendelwagen.svg" width="900pt">

Wir erhalten die Bewegungsgleichungen des Systems, indem wir wie gewöhnlich die folgenden Schritte ausführen:

 1. Quader und Pendel einzelnen freischneiden und _alle_ ässeren Kräfte eintragen
 2. Impuls- und Drallsätze (bzw. Spinsätze) für Quader und Pendel aufstellen
 3. Kinematische Relationen beschreiben
 4. Zwangskräfte und überzählige Koordinaten eliminieren

Beim Pendelwagen handelt es sich um ein System mit Freiheitsgrad $f=2$.
Die Bewegungsgleichungen lauten

<a id="EquationOfMotion"></a>
\begin{equation}\tag{1}
\begin{split}
    (m_S + m_P)\ddot{x} + m_P\,L\,\ddot{\varphi}\,\cos\varphi - m_P\,L\,\dot{\varphi}^2\,\sin\varphi - F_c(x) &= 0 \\[.5em]
    m_P\,L\,\ddot{x}\,\cos\varphi + m_P\,L^2\,\ddot{\varphi} + m_P\,g\,L\,\sin\varphi                         &= 0.
\end{split}
\end{equation}

Das ist _ein System von zwei nichtlinearen, gewöhnlichen Differentialgleichungen 2. Ordnung_.
Die Nichtlinearität entsteht durch die grosse Pendelauslenkung über die Winkelfunktionen $\cos\varphi$ und $\sin\varphi$ sowie durch das nichtlineare Federgesetz des Puffers $F_c(x)$.

### 1.2 Parameter

Für die Simulation der Dynamik werden wir folgende Parameter wählen:
| Grösse                  | Variable | Wert                   |
|:----------------------- |:--------:|:----------------------:|
| Masse Quader            |  $m_S$   | $2\,\mathrm{kg}$       |
| Masse Pendel            |  $m_P$   | $1\,\mathrm{kg}$       |
| Drucksteifigkeit Puffer |  $c$     | $1\,\mathrm{kN/m}$     |
| Entspannte Länge Puffer |  $l_0$   | $150\,\mathrm{mm}$     |
| Länge Pendel            |  $L$     | $150\,\mathrm{mm}$     |
| Breite Quader           |  $W$     | $500\,\mathrm{mm}$     | 
| Höhe Quader             | $H$     | $400\,\mathrm{mm}$     |
| Erdbeschleunigung       | $g$     | $9.81\,\mathrm{m/s^2}$ |

<span style="color:Orange">Definieren Sie die Modellparameter in [SI-Basiseinheiten](https://de.wikipedia.org/wiki/Internationales_Einheitensystem).</span>

In [ ]:
mS = <edit>                                                 # mass of cuboid [kg]
mP = <edit>                                                 # mass of pendulum [kg]
c = <edit>                                                  # spring constant [N/m]
l0 = <edit>                                                 # stress-free length of spring [m]
L = <edit>                                                  # length of pendulum [m]
W = <edit>                                                  # width of cuboid [m]
H = <edit>                                                  # height of cuboid [m]
g = <edit>                                                  # gravitational acceleration [m/s^2]

### 1.3 Nichtlineares Kraftgesetz

<span style="color:Orange">Definieren Sie die Funktion `Fc(x)` für das nichtlineare Federgesetz des Puffers.
Unter Druck soll die Federkraft $F_c$ propotional zur Auslenkung zunehmen; unter Zug kann der Puffer keine Kräfte aufnehmen.
</span>

> _**Tipps**_:
> * Definieren Sie die Funktion `Fc(x)` so, dass diese nicht nur für ein skalares `x`, sondern auch für einen Array `x` die Federkraft berechnet.
> * Nutzen Sie hierzu aus, dass NumPy bei [`(x < l0)`](https://numpy.org/doc/stable/reference/generated/numpy.less.html#numpy.less) einen boolschen Array der Grösse von `x` retourniert, das elementweise die Bedingungen `True` und `False` enthält.
> * Beim Rechnen mit boolschen Ausdrücken wird `True` als `1` und `False` als `0` interpretiert.


In [ ]:
Fc = lambda x: <edit>

Zur Verifikation der Funktion `Fc` plotten wir die Federkraft $F_c(x)$.

In [ ]:
fig2, ax = plt.subplots(1, 1, num="Federgesetz", clear=True)
_x = np.linspace(0, 3*l0, 100)                            # plot range
ax.plot(_x, Fc(_x), 'r')
ax.axvline(l0, color='k', linestyle='--', linewidth=1)
ax.set_xlabel('Auslenkung  $x$  [m]')
ax.set_ylabel('Federkraft  $F_c$  [N]');

### 1.4 Umformung von System 2. Ordnung in System 1. Ordnung (Zustandsform)

Die Differentialgleichungen $(1)$ beschreiben die Dynamik des Pendelwagens vollständig.
Das System der gewöhnlichen Differentialgleichungen liegen in $(1)$ jedoch noch in 2. Ordnung vor (zweifache Zeitableitung als höchste Ableitung).
Die numerischen Integratoren sind fast ausschliesslich für Anfangswertprobleme in 1. Ordnung implementiert.
Daher müssen wir das System $(1)$ also zuerst von 2. Ordnung in ein System 1. Ordnung umformen.
Das ist immer möglich, erfolgt immer gleich und wird in den folgenden **3 Schritten** gezeigt.

#### **Schritt 1:** System von Differentialgleichungen in Matrixform $\mathbf{M}\,\ddot{\mathbfit{q}} - \mathbfit{h} = \mathbfit{0}$ mit Massenmatrix $\mathbf{M}$

<span style="color:Orange">Schreiben Sie das System von $f$ Gleichungen $(1)$ 
\begin{equation}\tag{1}
\begin{split}
    (m_S + m_P)\ddot{x} + m_P\,L\,\ddot{\varphi}\,\cos\varphi - m_P\,L\,\dot{\varphi}^2\,\sin\varphi - F_c(x) &= 0 \\[.5em]
    m_P\,L\,\ddot{x}\,\cos\varphi + m_P\,L^2\,\ddot{\varphi} + m_P\,g\,L\,\sin\varphi                         &= 0
\end{split}
\end{equation}
in die Matrixform
\begin{equation}\tag{2}
    \mathbf{M}(\mathbfit{q})\,\ddot{\mathbfit{q}} - \mathbfit{h}(t, \mathbfit{q}, \dot{\mathbfit{q}}) = \mathbfit{0}
\end{equation}
um.
Darin definiert $\mathbfit{q} = \begin{pmatrix}x \\ \varphi\end{pmatrix} \in \mathbb{R}^f$ den Vektor der Lagekoordinaten (die sogenannten Minimalkoordinaten).
Die Massenmatrix $\mathbf{M}(\mathbfit{q}) \in \mathbb{R}^{f,f}$ enthält alle Vorfaktoren der Beschleunigungsterme $\ddot{x}$ und $\ddot{\varphi}$.
Diese Massenmatrix $\mathbf{M}$ ist symmetrisch und selbst eine Funktion der Minimalkoordinaten $\mathbfit{q}$.
Der Vektor $\mathbfit{h}(t, \mathbfit{q}, \dot{\mathbfit{q}})\in \mathbb{R}^f$ vereinigt alle verbleibenden Terme ohne die Beschleunigungen $\ddot{x}$ und $\ddot{\varphi}$.
</span>

\begin{equation}\tag{3}
    \begin{pmatrix} <edit> & <edit> \\ <edit> & <edit> \end{pmatrix}
    \begin{pmatrix}\ddot{x} \\ \ddot{\varphi}\end{pmatrix}
    - \begin{pmatrix} <edit> \\ <edit> \end{pmatrix}
    = \begin{pmatrix} 0 \\ 0 \end{pmatrix}
\end{equation}

<span style="color:Orange">Definieren Sie die (symmetrische) Massenmatrix `M(q)` als Funktion von `q`.</span>

In [ ]:
def M(q):
    """Mass matrix"""
    
    # Parse variables
    x, φ = q
    
    return np.array([
        [        <edit>,         <edit>],
        [        <edit>,         <edit>]
        ])

<span style="color:Orange">Definieren Sie den Vektor `h(t, q, q_t)` als Funktion von `t`, `q` und `q_t`.</span>

> _**Hinweis**_: Um Zeitableitungen zu kennzeichnen, hängen wir der Variablen ein `_t` an. Wir schreiben für $\dot{\mathbfit{q}}$ also `q_t`.

In [ ]:
def h(t, q, q_t):
    """Vector of cyroscopic and active forces"""
    
    # Parse variables    
    x, φ = q
    x_t, φ_t = q_t
    
    return np.array([
                                <edit>,
                                <edit>
        ])

#### **Schritt 2:** Differentialgleichung 2. Ordnung _explizit_ in der Form $\ddot{\mathbfit{q}} = \mathbfit{a}(t, \mathbfit{q}, \dot{\mathbfit{q}})$

Die Gleichung $(2)$ ist eine _implizite_ Gleichung für die Beschleunigung $\ddot{\mathbfit{q}}$.
Für die numerische Integration benötigen wir die Gleichung für die Beschleunigung in _expliziter_ Form, nämlich
\begin{equation}\tag{4}
    \ddot{\mathbfit{q}} = \mathbf{M}^{-1}(\mathbfit{q}) \, \mathbfit{h}(t, \mathbfit{q}, \dot{\mathbfit{q}}) =: \mathbfit{a}(t, \mathbfit{q}, \dot{\mathbfit{q}}).
\end{equation}
Darin definiert die Vektorfunktion $\mathbfit{a}(t, \mathbfit{q}, \dot{\mathbfit{q}})$ unsere Differentialgleichung 2. Ordnung, eben in expliziter Form $\ddot{\mathbfit{q}} = \mathbfit{a}(t, \mathbfit{q}, \dot{\mathbfit{q}})$.

<span style="color:Orange">Definieren Sie die den Vektorfunktion `a(t, q, q_t)`, die die Beschleunigung $\ddot{\mathbfit{q}}$ _explizit_ als Funktion von `t`, `q` und `q_t` beschreibt.</span>

> _**Tipp**_:
> * In der Numerik wird nur in Ausnahmefällen die Inverse $\mathbf{M}^{-1}(\mathbfit{q})$ berechnet. Statt dessen wird meistens das Gleichungssystem $\mathbf{M}\,\ddot{\mathbfit{q}} = \mathbfit{h}$ numerisch nach $\ddot{\mathbfit{q}}$ gelöst. Verwenden Sie hierzu den Gleichungslöser [`np.linalg.solve()`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).

In [ ]:
def a(t, q, q_t):
    r"""Differential equation in second-order form (acceleration)
    
        q_tt = a(t, q, q_t) := M(q)\h(t, q, q_t)
    """
    
    return <edit>

#### Verständnisfragen

<span style="color:Orange">Ohne die Funktion `a(t, q, q_t)` aufzurufen: Welche Rückgabewerte erwarten Sie für die Funktion `a(t, q, q_t)` bei folgenden Eingabewerten:</span>

> _**Tipp:**_ Stellen Sie sich die Situation anhand der Systemskizze bildlich vor.  Welche Beschleunigungen $\ddot{x}$ und $\ddot{\varphi}$ treten für den gegebenen Zustand ein?

| Fall | Eingabewerte                                     | erwarteter Rückgabewert        |
|:----:|:------------------------------------------------ |:-------------------------------|
|  (i) | `t = 3`, `q = [3*l0,       0]`, `q_t = [-1, 0]`  | `a = [<edit>, <edit>]`         |
| (ii) | `t = 0`, `q = [3*l0, np.pi/2]`, `q_t = [ 0, 0]`  | `a = [<edit>, <edit>]`         |

<span style="color:Orange">Überprüfen Sie Ihre Erwartung mit dem effektiven Resultat.</span>

In [ ]:
for args in (( '(i)', 3,       [3*l0, 0], [-1, 0]),
             ('(ii)', 0, [3*l0, np.pi/2],  [0, 0])):
    case, t, q, q_t = args
    print(f"{case:>4}   a = {a(t, q, q_t)}")

#### **Schritt 3:** Differentialgleichung 1. Ordnung in der Form $\dot{\mathbfit{y}} = \mathbfit{f}(t, \mathbfit{y})$ mit Zustandsvektor $\mathbfit{y}=\begin{pmatrix} \mathbfit{q} \\ \dot{\mathbfit{q}} \end{pmatrix}$

_**Trick:**_ Definiere neue Variable für die Geschwindigkeit $\mathbfit{z} := \dot{\mathbfit{q}}$.
Damit lässt sich $(4)$ als ein System von Differentialgleichungssystemen
\begin{equation}\tag{5}
\begin{split}
    \dot{\mathbfit{q}}  &= \mathbfit{z} \\
    \ddot{\mathbfit{q}} &= \mathbfit{a}(t, \mathbfit{q}, \mathbfit{z})
\end{split}
\end{equation}
ausdrücken.
Büscheln wir das in Matrizen, lässt es sich schreiben als
\begin{equation}\tag{6}
    \underbrace{\begin{bmatrix} \dot{\mathbfit{q}} \\ \ddot{\mathbfit{q}} \end{bmatrix}}_{\dot{\mathbfit{y}}}
    = \underbrace{\begin{bmatrix} \mathbfit{z} \\ \mathbfit{a}(t, \mathbfit{q}, \mathbfit{z}) \end{bmatrix}}_{\mathbfit{f}(t, \mathbfit{y})},
\end{equation}
oder kurz
\begin{equation}\tag{6'}
\dot{\mathbfit{y}} = \mathbfit{f}(t, \mathbfit{y}).
\end{equation}
Das ist die Zustandsform der Bewegungsgleichungen $(1)$.
Darin definiert $\mathbfit{f}(t, \mathbfit{y})$ die Zustandsänderung  $\dot{\mathbfit{y}} \in \mathbb{R}^{2f}$ als Funktion der Zeit $t$ und des aktuellen Zustands $\mathbfit{y} \in \mathbb{R}^{2f}$.

> _**Hinweis**_: Die Zahl der Gleichungen in $(6)$ hat sich gegenüber $(1)$ von $f$ auf $2f$ verdoppelt, dafür hat sich die Ordnung von 2 auf 1 reduziert.

<span style="color:Orange">Definieren Sie die den Vektorfunktion `f(t, y)`, die die Zustandsänderung $\dot{\mathbfit{y}}$ _explizit_ als Funktion von `t`, und `y` beschreibt.</span>

> _**Tipps**_:
> * Verwenden Sie die Funktion [`np.split()`](https://numpy.org/doc/stable/reference/generated/numpy.split.html#numpy.split) um den Zustandsvektor `y` in die Vektoren für die Lage `q` und Geschwindigkeit `q_t` aufzuteilen.
> * Verwenden Sie die Funktion [`np.concatenate()`](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html#numpy.concatenate) um den Zustandsvektor `y` aus den Vektoren der Lage `q` und Geschwindigkeit `q_t` zusammenzusetzen.

In [ ]:
def f(t, y):
    """Differential equation in first-order form (state)
    
        y_t = f(t, y)       with y := [q, q_t]
    """
    
    q, q_t = <edit>
    
    return <edit>

Die Bewegungsgleichungen sind mit der Funktion `f(t, y)` nun vollständig definiert und können für einen Anfangszustand `y0` numerisch gelöst werden.

#### Verständnisfragen

<span style="color:Orange">Ohne die Funktion `f(t, y)` aufzurufen: Welche Rückgabewerte erwarten Sie für die Funktion `f(t, y)` bei folgenden Eingabewerten:</span>

| Fall | Eingabewerte                           | erwarteter Rückgabewert                 |
|:----:|:-------------------------------------- |:----------------------------------------|
|  (i) | `t = 3`, `y = [3*l0,       0, -1, 0]`  | `y_t = [<edit>, <edit>, <edit>, <edit>]`|
| (ii) | `t = 0`, `y = [3*l0, np.pi/2,  0, 0]`  | `y_t = [<edit>, <edit>, <edit>, <edit>]`|

### 1.5 Lösung des Anfangswertproblems (explizite numerische Zeitintegration)

Wir lösen nun die Bewegungsgleichungen mit dem Anfangszustand

| Variable          | Wert                |
|:-----------------:|--------------------:|
| $x_0$             | $5\,l_0$            |
| $\varphi_0$       | $0\,\mathrm{rad/s}$ |
| $\dot{x}_0$       | $-1\,\mathrm{m/s}$  |
| $\dot{\varphi}_0$ | $0\,\mathrm{rad/s}$ |

Die Gleichungen in Zustandsform $(6)$ werden über den Zeitbereich $[0, T]$ integriert mit $T = 5\,\mathrm{s}$.
Wir verwenden [`solve_ivp()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) zur explizite Zeitintegration mit dem Runge–Kutta-Verfahren.

In [ ]:
T = 5                                                       # time interval to solve
x0, φ0 = 5*l0, np.radians(0)                                # initial position
x0_t, φ0_t = -1, 0                                          # initial velocity

In [ ]:
# Stack initial state vector
q0 = np.array([x0, φ0])                                     # initial position
q0_t = np.array([x0_t, φ0_t])                               # initial velocity
y0 = np.concatenate((q0, q0_t))                             # initial state

# Solve initial value problem
sol = solve_ivp(f, [0, T], y0, max_step=np.inf)

# Parse output
_t = sol.t
_y = sol.y

Interpoliere die Lösung auf äquidistante Stützstellen mit feiner Auflösung (für glatte Plotverläufe und Animationen)

In [ ]:
N = 300
t = np.linspace(0, T, N)
y = CubicSpline(_t, _y, axis=1)(t)

Extrahiere Lagen und Geschwindigkeiten und berechne Beschleunigungen

In [ ]:
q, q_t = np.split(y, 2)
q_tt = np.array([a(*args) for args in zip(t, q.T, q_t.T)]).T

x, φ = q                                                    # positions
x_t, φ_t = q_t                                              # velocities
x_tt, φ_tt = q_tt                                           # accelerations

#### Verständnisfragen

<span style="color:Orange">Bevor Sie die Lösung anschauen und animieren: Welchen Bewegungsablauf erwarten Sie für den gegebenen Anfangszustand? Welche Bewegungsfolge stellt sich ein? Was wird wann passieren?</span>

### 1.6 Plot und Animation

Wir stellen die berechneten Lagen und Geschwindigkeiten als Zeitverläufe dar.

In [ ]:
fig3, axs = plt.subplots(3, 2, num='Zeitverläufe', clear=True, sharex=True, figsize=(12, 10))
axs[0, 0].plot(t, x, 'g')
axs[0, 0].set_ylabel(r'Lage  $x$  $\mathrm{[m]}$')

axs[1, 0].plot(t, x_t, 'b')
axs[1, 0].set_ylabel(r'Geschwindigkeit  $\dot{x}$  $\mathrm{[m/s]}$')

axs[2, 0].plot(t, x_tt, 'r')
axs[2, 0].set_xlabel(r'Zeit  $t$  [s]')
axs[2, 0].set_ylabel(r'Beschleunigung  $\ddot{x}$  $\mathrm{[m/s]}$')

axs[0, 1].plot(t, np.degrees(φ), 'g')
axs[0, 1].set_xlabel(r'Zeit  $t$  [s]')
axs[0, 1].set_ylabel(r'Orientierung  $\varphi$  $\mathrm{[°]}$')

axs[1, 1].plot(t, φ_t, 'b')
axs[1, 1].set_ylabel(r'Winkelgeschwindigkeit  $\dot{\varphi}$  $\mathrm{[rad/s]}$');

axs[2, 1].plot(t, φ_tt, 'r')
axs[2, 1].set_xlabel(r'Zeit  $t$  [s]')
axs[2, 1].set_ylabel(r'Winkelbeschleunigung  $\ddot{\varphi}$  $\mathrm{[rad/s^2]}$');

Wir stellen den berechneten Verlauf der Kontaktkraft $F_c$ des Puffers dar.

In [ ]:
fig4, ax = plt.subplots(1, 1, num='Kraftverlauf', clear=True, sharex=True, figsize=(10, 6))
ax.plot(t, Fc(x), 'r')
ax.set_xlabel(r'Zeit  $t$  [s]')
ax.set_ylabel(r'Kraft  $F_c$  [N]');

Zeichne Szene mit Wand, Pendelwagen und Puffer in der Ausgangslage $\mathbfit{q}_0$

In [ ]:
fig5, ax = plt.subplots(1, 1, num='Animation', clear=True, 
                        figsize=(15, 5))
# Wall
Δ = W/10                                                    # hatch width
lineW = ax.plot([7*W, 0, 0], [0, 0, H], color='k', linewidth=2)
polyW = ax.fill([7*W, 0, 0, -Δ, -Δ, 7*W], [0, 0, H, H, -Δ, -Δ], hatch='//', facecolor='w')

# Cuboid
rectS = plt.Rectangle(xy=(x[0], 0), width=W, height=H, edgecolor='k', facecolor=('k', .2), linewidth=2)
ax.add_patch(rectS)

# Pendulum
lineP, = ax.plot([x[0] + W/2, x[0] + W/2 + L*np.sin(φ[0])], [H/2, H/2 - L*np.cos(φ[0])], 'ko-', 
                 markersize=5, linewidth=3, markerfacecolor='w', markeredgecolor='k')
pointP, = ax.plot([x[0] + W/2 + L*np.sin(φ[0])], [H/2 - L*np.cos(φ[0])], 'ko', markersize=15)

# Buffer
lineB, = ax.plot([0, min(x[0], l0)], [H/2, H/2], 'r-', linewidth=5, alpha=.5)

ax.set_xlim([-Δ, 7*W])
ax.set_aspect('equal')
ax.axis('off');

Animieren der Szene für die Bewegung des Pendelwagens.

In [ ]:
def update(frame):
    """Update function for animation"""
    
    # for each frame, update the data stored on each artist.
    x, φ = q[:,frame]
    
    rectS.set_x(x)
    lineP.set_data([x + W/2,  x + W/2 + L*np.sin(φ)], [H/2, H/2 - L*np.cos(φ)])
    pointP.set_data([x + W/2 + L*np.sin(φ)], [H/2 - L*np.cos(φ)])
    lineB.set_data([0, min(x, l0)], [H/2, H/2])
    
    return (rectS, lineP, pointP)


ani = animation.FuncAnimation(fig=fig5, func=update, frames=len(t), interval=T/N*1e3, blit=True)

HTML(ani.to_jshtml())

#### Verständnisfragen

<span style="color:Orange">Verhält sich das berechnete System Ihrer Meinung nach physikalisch richtig?</span>

### 1.7 Alternative Anfangszustände

<span style="color:Orange">Spielen Sie mit unterschiedlichen Anfangszuständen. Wie verhält sich das gleiche System bei unterschiedlichen Anfangswerten für Lage und Geschwindigkeit?</span>

<span style="color:Orange">Versuchen Sie Anfangszustände zu finden, um folgende Bewegungen zu erhalten:</span>
 1. <span style="color:Orange">Das Pendel vollführt nach dem Auftreffen auf den Puffer eine kontinuierlich überschlagende Bewegung.</span>
 2. <span style="color:Orange">Der Pendelwagen entfernet sich erst vom Puffer, ehe er nochmals auf den Puffer aufschlägt.</span>
 3. ...


### 1.8 Verifikation

Auf den hier modellierten Pendelwagen mit Puffer wirken weder aktive äusseren Kräfe noch gibt es irgendwelche dissipativen Elemente (Reibung oder Dämpfung).
Thermodynamisch betrachtet muss für das System die Energieerhaltung gelten, d.h. die Gesammtenergie $E$ aus Summe der kinetischen Energien $T_i$ und potentiellen Energien $V_j$ muss erhalten/konstant bleiben, also
\begin{equation}\tag{E}
    E := \sum_i T_i + \sum_j V_j \stackrel{!}{=} \mathrm{const}.
\end{equation}
Wir können die Energieerhaltung ausnutzen, um die Plausibilität der berechneten Lösung zu verifizieren.

In [ ]:
o = np.zeros_like(t)                                        # zero vector
vS = np.stack((x_t, o, o)).T                                # velocity of cuboid S
Ω = np.stack((o, o, φ_t)).T                                 # angular velocity of pendulum
r_SP = np.stack((L*np.sin(φ), -L*np.cos(φ), o)).T           # distance vector from S to P
vP = vS + np.cross(Ω, r_SP)                                 # velocity of pendulum P
r_OS = np.stack((x + W/2, o + H/2, o)).T                    # position of S
r_OP = r_OS + r_SP                                          # position of P

In [ ]:
T1 = mS*np.vecdot(vS,vS)/2                                  # kinetic energy of cuboid
T2 = mP*np.vecdot(vP,vP)/2                                  # kinetic energy of pendulum
V1 = Fc(x)*(l0 - x)/2                                       # potential energy of buffer
V2 = np.vecdot(mP*g*r_OP,[0, 1, 0])                         # potential energy of pendulum

In [ ]:
fig6, ax = plt.subplots(1, 1, num='Energy Diagramm', clear=True)
ax.stackplot(t, V1, V2, T1, T2, 
             labels=['potentielle Energy Puffer', 'potentielle Energy Pendel', 'kinetische Energy Quader', 'kinetische Energy Pendel'],
             colors=['r', 'g',  ('b', .8), ('b', .4)]);
ax.legend(loc='lower right');

<span style="color:Orange">Wie gut ist die Energieerhaltung $(\mathrm{E})$ bei der numerischen Integration erfüllt?</span>

<span style="color:Orange">Setzen Sie bei der numerischen Integration mit [`solve_ivp()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) die maximale Schrittweite auf `max_step = 0.01`. Wie verändert sich das Resultat?</span>

## Résumé

In dieser Aufgabe haben Sie gelernt, die Bewegungsgleichungen eines dynamischen Systems mit numerischen Methoden zu lösen.
Der gezeigte Lösungsprozess ist prinzipiell immer gleich und kann von diesem Notebook für andere dynamische Probleme übernommen werden.
Die grösste Herausforderung stellt die Modellbildung und die Herleitung der Bewegungsgleichungen dar.
Die Herleitung der Bewegungsgleichungen (ausgehend von einem Modell über Freischneiden, Impuls- und Spinsätze, Kinematik, Kraftgesetze und Elimination der Zwangskräfte) ist ein wesentliches Lernziel von `MT.PHY3` Maschinendynamik.

## A. Alternative Herleitung der Bewegungsgleichungen

Es gibt in der Mechanik Methoden um die Bewegungsgleichungen von Mehrkörpersystemen direkt abzuleiten, **ohne dass die einzelnen Körper freigeschnitten werden müssen**.
Leider können wir in `MT.PHY3` Maschinendynamik nicht (vertieft) auf diese Methoden eingehen, obwohl sie von grossem praktischen Nutzen sind.
Im Folgenden wird anhand unseres Pendelwagens exemplarisch gezeigt, wie mit dem [Lagrange-Formalismus](https://de.wikipedia.org/wiki/Lagrange-Formalismus) und mit den projizierten Newton–Euler-Gleichungen die Bewegungsgleichungen $(1)$ hergeleitet werden.

### A.1 Lagrange-Formalismus

Die Bewegungsgleichungen eines Systems starrer Körper, die ausschliesslich [holonome Bindungen](https://de.wikipedia.org/wiki/Holonom) unterliegen, lassen sich direkt mit den [Lagrange-Gleichungen 2. Art](https://de.wikipedia.org/wiki/Lagrange-Formalismus) angeben
\begin{equation}\tag{L}
    \boxed{\frac{\mathrm{d}}{\mathrm{d}t}\left(\frac{\partial T}{\partial\dot{\mathbfit{q}}}\right)^\top - \left(\frac{\partial T}{\partial\mathbfit{q}}\right)^\top + \left(\frac{\partial V}{\partial\mathbfit{q}}\right)^\top = \mathbfit{f}^\mathrm{NP}}\ .
\end{equation}
Darin definiert $T(t, \mathbfit{q}, \dot{\mathbfit{q}})$ die gesammte _kinetische Energie_ des Systems, $V(t, \mathbfit{q})$ die gesammte _potentielle Energie_ und $\mathbfit{f}^\mathrm{NP}$ die verbleibenden, aktiven _Nichtpotentialkräfte_.
Die Bewegungsgleichungen werden also gewonnen, indem die kinetischen und potentiellen gemäss Vorschrift $(\mathrm{L})$ nach den Minimalkoordinaten $\mathbfit{q}$ und der Zeitableitung $\dot{\mathbfit{q}}$ partiell abgeleitet werden.
Das Freischneiden der Körper mit Einführen von Zwangskräften, das Aufstellen der Impuls- und Drallsätze und das anschliessende Eliminieren der Zwangskräfte entfällt bei der Anwendung des Lagrange-Formalismus.
Im Folgenden zeigen wir schrittweise das Vorgehen am Beispiel des Pendelwagens mit Puffer.

Die kinetische Energie unseres Pendelwagen summiert sich aus den kinetischen Energien von Quader und Pendel
\begin{equation}
    T(t, \mathbfit{q}, \dot{\mathbfit{q}}) = \sum_i T_i = \frac{1}{2}\,m_S\,\mathbfit{v}_S^\top\mathbfit{v}_S + \frac{1}{2}\,m_P\,\mathbfit{v}_P^\top\mathbfit{v}_P,
\end{equation}
mit
\begin{equation}
    \mathbfit{v}_S = \begin{pmatrix} \dot{x} \\ 0 \\ 0 \end{pmatrix}, \quad \mathbfit{v}_P = \begin{pmatrix} \dot{x} + L\,\dot{\varphi}\,\cos\varphi \\ L\,\dot{\varphi}\,\sin\varphi \\ 0 \end{pmatrix}.
\end{equation}
Damit resultiert
\begin{equation}
    T = \frac{1}{2}(m_S + m_P)\,\dot{x}^2 + m_P\,L\,\dot{x}\,\dot{\varphi}\,\cos\varphi + \frac{1}{2}m_P\,L^2\,\dot{\varphi}^2.
\end{equation}

Die potentielle Energie setzt sich zusammen aus der potentielle Energie der Feder $F_c$ und dem Gravitationspotential des Pendels
\begin{equation}
    V(t, \mathbfit{q}) = \sum_j V_j = -\int_{l_0}^x{F_c(\bar{x})\,\mathrm{d}\bar{x}} - m_P\,g\,L\,\cos\varphi .
\end{equation}

Das Federpotential hat ein negatives Vorzeichen, weil die Federkraft $F_c$ als Druckkraft interpertiert wird.
Das negative Vorzeichen beim Gravitationspotential kommt daher, da $\varphi$ die Auslenkung des Pendels aus der unteren Gleichgewichtslage aus misst.

Beim Pendelwagen können alle Kräfte über das Potential $V$ beschrieben werden.
Daher setzen wir die Nichtpotentialkräfte $\mathbfit{f}^\mathrm{NP} = \mathbfit{0}$.

Führen wir die Ableitungen gemäss $(\mathrm{L})$ komponentenweise für $\mathbfit{q} = \begin{pmatrix} x \\ \varphi \end{pmatrix}$ aus

\begin{align}
    0 \stackrel{!}{=}& \frac{\mathrm{d}}{\mathrm{d}t}\frac{\partial T}{\partial\dot{x}} - \frac{\partial T}{\partial x} + \frac{\partial V}{\partial x} = 
    (m_S + m_P)\ddot{x} + m_P\,L\,\ddot{\varphi}\,\cos\varphi -  m_P\,L\,\dot{\varphi}^2\,\sin\varphi - 0 - F_c(x) \\[1em]
    0 \stackrel{!}{=}& \frac{\mathrm{d}}{\mathrm{d}t}\frac{\partial T}{\partial\dot{\varphi}} - \frac{\partial T}{\partial \varphi} + \frac{\partial V}{\partial \varphi} = 
    m_P\,L\,\ddot{x}\,\cos\varphi - m_P\,L\,\dot{x}\,\dot{\varphi}\,\sin\varphi + m_P\,L^2\ddot{\varphi} + m_P\,L\,\dot{x}\,\dot{\varphi}\,\sin\varphi + m_P\,g\,L\,\sin\varphi
\end{align}

so erhalten wir — wie durch magische Hand — die [Bewegungsgleichungen aus $(1)$](#EquationOfMotion)

\begin{equation}\tag{1}
\begin{split}
    (m_S + m_P)\ddot{x} + m_P\,L\,\ddot{\varphi}\,\cos\varphi - m_P\,L\,\dot{\varphi}^2\,\sin\varphi - F_c(x) &= 0 \\[.5em]
    m_P\,L\,\ddot{x}\,\cos\varphi + m_P\,L^2\,\ddot{\varphi} + m_P\,g\,L\,\sin\varphi                         &= 0.
\end{split}
\end{equation}

### A.2 Projizierte Newton–Euler-Gleichungen

Eine noch mächtigere Alternative zur Energiemethode mit dem Lagrange-Formalismus ist die Betrachtung der Bewegungsgleichungen als projizierte Newton–Euler-Gleichungen.
Bei diesem Formalismus werden, wie der Name impliziert, die Impuls- und Spinsätze eines Systems auf die $f$ freien Bewegungsrichtungen projiziert.
Für ein System aus $N$ starren Körpern lauten die projizierten Newton–Euler-Gleichungen
\begin{equation}\tag{NE}
    \boxed{\mathbfit{0} = \sum_{i = 1}^{N} \left[\left(\mathbf{J}_S^\top\dot{\mathbfit{p}} + \mathbf{J}_R^\top\dot{\mathbfit{N}}_S\right)
        -  \left(\mathbf{J}_Q^\top\mathbfit{F}^\mathrm{a} + \mathbf{J}_R^\top\mathbfit{M}_Q^\mathrm{a} \right) \right]_i }\ .
\end{equation}
Darin sind körperweise $\mathbf{J}_S := \frac{\partial\mathbfit{v}_S}{\partial\dot{\mathbfit{q}}}$ die Jakobi-Matrix der Translation im Massenmittelpunkt $S$, $\dot{\mathbfit{p}}$ die Impulsänderung, $\mathbf{J}_R := \frac{\partial\mathbfit{\Omega}}{\partial\dot{\mathbfit{q}}}$ die Jakobi-Matrix der Rotation, $\dot{\mathbfit{N}}_S$ die Spinänderung, $\mathbf{J}_Q := \frac{\partial\mathbfit{v}_Q}{\partial\dot{\mathbfit{q}}}$ die Jakobi-Matrix der Translation im Bezugspunkt $Q$, $\mathbfit{F}^\mathrm{a}$ die aktive äussere Kraft und $\mathbfit{M}_Q^\mathrm{a}$ das aktive äussere Moment bezüglich $Q$.
Zwangskräfte sind passiv da ihre Projizierten verschwinden und müssen daher nicht in $\mathrm{(NE)}$ nicht berücksichtigt werden.
Im Folgenden zeigen wir schrittweise das Vorgehen am Beispiel des Pendelwagens mit Puffer.

#### Kinematik

Für die Projektion der Impulssätze benötigen wir die Jakobi-Matrizen der Translation in den Punkten $S$ und $Q$.
Diese gewinnen wir, indem wir die Geschwindigkeit in Bezugspunkt bestimmen und dann die Minimalgeschwindigkeiten $\dot{\mathbfit{q}}$ «ausklammern».
Für den Punkt $S$ erhalten wir so
\begin{equation}
    \mathbfit{v}_S = \begin{pmatrix} \dot{x} \\ 0 \\ 0 \end{pmatrix} 
        \equiv \begin{pmatrix} 1 & 0 \\ 0 & 0 \\ 0 & 0 \end{pmatrix} \begin{pmatrix} \dot{x} \\ \dot{\varphi} \end{pmatrix}
    \quad\Rightarrow\quad \mathbf{J}_S = \begin{pmatrix} 1 & 0 \\ 0 & 0 \\ 0 & 0 \end{pmatrix},
\end{equation}
und für Punkt $P$
\begin{equation}
    \mathbfit{v}_P = \mathbfit{v}_S + \mathbfit{\Omega}\times\mathbfit{r}_{SP} = \begin{pmatrix} \dot{x} + L\,\dot{\varphi}\,\cos\varphi \\ L\,\dot{\varphi}\,\sin\varphi \\ 0 \end{pmatrix} 
        \equiv \begin{pmatrix} 1 & L\,\cos\varphi \\ 0 & L\,\sin\varphi \\ 0 & 0 \end{pmatrix} \begin{pmatrix} \dot{x} \\ \dot{\varphi} \end{pmatrix}
    \quad\Rightarrow\quad \mathbf{J}_P = \begin{pmatrix} 1 & L\,\cos\varphi \\ 0 & L\,\sin\varphi \\ 0 & 0 \end{pmatrix}.
\end{equation}

#### Quader (Körper $1$)

Der Impuls und die Impulsänderung des Quaders sind
\begin{equation}
    \mathbfit{p}_S = m_S\,\mathbfit{v}_S = m_S\begin{pmatrix} \dot{x} \\ 0 \\ 0 \end{pmatrix} 
    \quad\Rightarrow\quad \dot{\mathbfit{p}}_S = m_S\begin{pmatrix} \ddot{x} \\ 0 \\ 0 \end{pmatrix}.
\end{equation}
Beim Quader greift nur die Federkraft $F_c$ als aktive äussere Kraft an, also ist
\begin{equation}
    \mathbfit{F}_S = \begin{pmatrix} F_c(x) \\ 0 \\ 0 \end{pmatrix}.
\end{equation}

#### Pendel (Körper $2$)

Die Masse $m_P$ des Pendels ist im Punkt $P$ konzentriert.
Der Impuls und die Impulsänderung des Pendels sind demnach
\begin{equation}
    \mathbfit{p}_P = m_P\,\mathbfit{v}_P = m_P\begin{pmatrix} \dot{x} + L\,\dot{\varphi}\,\cos\varphi \\ L\,\dot{\varphi}\,\sin\varphi \\ 0 \end{pmatrix} 
    \quad\Rightarrow\quad \dot{\mathbfit{p}}_P = m_P\begin{pmatrix} \ddot{x} + L\,\ddot{\varphi}\,\cos\varphi - L\,\dot{\varphi}^2\sin\varphi \\ L\,\ddot{\varphi}\,\sin\varphi + L\,\dot{\varphi}^2\cos\varphi \\ 0 \end{pmatrix}
\end{equation}
Beim Pendel greift die Gewichtskraft $m_P\,g$ als einzige aktive äussere Kraft an, also ist
\begin{equation}
    \mathbfit{F}_P = \begin{pmatrix} 0 \\ -m_P\,g \\ 0 \end{pmatrix}.
\end{equation}

#### Bewegungsgleichungen
Beide Körper haben keinen Spin und es wirkt kein aktives äusseres Moment.
Die projizierten Newton–Euler-Gleichungen des Pendelwagens reduzieren sich also auf
\begin{equation}
    \mathbfit{0} = \mathbf{J}_S^\top \left(\dot{\mathbfit{p}}_S - \mathbfit{F}_S\right) + \mathbf{J}_P^\top \left(\dot{\mathbfit{p}}_P - \mathbfit{F}_P\right).
\end{equation}

Werten wir diese aus, erhalten wir — wiederum wie durch magische Hand — die [Bewegungsgleichungen aus $(1)$](#EquationOfMotion)
\begin{equation}\tag{1}
\begin{split}
    (m_S + m_P)\ddot{x} + m_P\,L\,\ddot{\varphi}\,\cos\varphi - m_P\,L\,\dot{\varphi}^2\,\sin\varphi - F_c(x) &= 0 \\[.5em]
    m_P\,L\,\ddot{x}\,\cos\varphi + m_P\,L^2\,\ddot{\varphi} + m_P\,g\,L\,\sin\varphi                         &= 0.
\end{split}
\end{equation}